# Interactive Anatomix Feature Comparison

This notebook allows for interactive comparison of Anatomix features (v1.0, v1.2, v1.3) between MRI and CT across different regions.
1. Select a version and region.
2. Extract features and perform PCA.
3. Click on the CT image to see cosine similarity in both MR and CT features.

In [1]:
%matplotlib widget
import os
import sys
import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.decomposition import PCA
import ipywidgets as widgets
from IPython.display import display

# Add project root to path for imports
sys.path.append(os.path.abspath(".."))
from anatomix.model.network import Unet

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA_ROOT = "/gpfs/accounts/jjparkcv_root/jjparkcv98/minsukc/MRI2CT/SynthRAD_combined/1.5mm_registered_flat"

REGIONS = {"Head": "1HNA001", "Thorax": "1THA001", "Abdomen": "1ABA005", "Pelvis": "1PA001"}

In [2]:
def clean_state_dict(state_dict):
    """Strip _orig_mod prefix from torch.compile"""
    new_state_dict = {}
    for k, v in state_dict.items():
        name = k.replace("_orig_mod.", "")
        new_state_dict[name] = v
    return new_state_dict


def load_anatomix_model(version, device):
    print(f"Loading Anatomix {version}...")
    if version == "v1.0":
        model = Unet(3, 1, 16, 4, 16).to(device)
        ckpt = "/home/minsukc/MRI2CT/anatomix/model-weights/anatomix.pth"
    elif version in ["v1.2", "v1.3"]:
        model = Unet(3, 1, 16, 5, 20, norm="instance", interp="trilinear", pooling="Avg").to(device)
        if version == "v1.2":
            ckpt = "/home/minsukc/MRI2CT/anatomix/model-weights/best_val_net_G_v1_2.pth"
        else:  # v1.3
            ckpt = "/home/minsukc/MRI2CT/anatomix/model-weights/best_val_net_G_real_v1_3.pth"
    else:
        raise ValueError(f"Unknown version: {version}")

    if os.path.exists(ckpt):
        state_dict = torch.load(ckpt, map_location=device)
        if "model_state_dict" in state_dict:
            state_dict = state_dict["model_state_dict"]
        model.load_state_dict(clean_state_dict(state_dict), strict=True)
        print(f"Loaded weights from {ckpt}")
    else:
        print(f"WARNING: Weights NOT FOUND at {ckpt}")

    model.eval()
    return model


def extract_features(model, volume_np, device):
    with torch.no_grad():
        inp = torch.from_numpy(volume_np[None, None]).float().to(device)
        inp = (inp - inp.min()) / (inp.max() - inp.min() + 1e-8)

        _, _, D, H, W = inp.shape
        pad_d = (32 - (D % 32)) % 32
        pad_h = (32 - (H % 32)) % 32
        pad_w = (32 - (W % 32)) % 32

        padding = (0, pad_w, 0, pad_h, 0, pad_d)
        inp_padded = F.pad(inp, padding, mode="reflect")
        feats_padded = model(inp_padded)
        feats = feats_padded[:, :, :D, :H, :W]

        return feats.squeeze(0).cpu().numpy()


def project_pca(feats, pca_obj=None):
    C, D, H, W = feats.shape
    X = feats.reshape(C, -1).T
    if pca_obj is None:
        pca_obj = PCA(n_components=3)
        idx = np.random.choice(X.shape[0], 25000, replace=False)
        pca_obj.fit(X[idx])

    Y = pca_obj.transform(X)
    Y_img = Y.reshape(D, H, W, 3)
    vmin = Y_img.min(axis=(0, 1, 2))
    vmax = Y_img.max(axis=(0, 1, 2))
    Y_norm = (Y_img - vmin) / (vmax - vmin + 1e-8)
    return np.clip(Y_norm, 0, 1), pca_obj

In [3]:
ver_dropdown = widgets.Dropdown(options=["v1.0", "v1.2", "v1.3"], value="v1.3", description="Version:")
region_dropdown = widgets.Dropdown(options=list(REGIONS.keys()), value="Abdomen", description="Region:")
load_btn = widgets.Button(description="Load Data & Features")
output = widgets.Output()

current_data = {}


def on_load_clicked(b):
    with output:
        output.clear_output()
        ver = ver_dropdown.value
        region = region_dropdown.value
        subj_id = REGIONS[region]

        print(f"Loading {region} ({subj_id}) with {ver}...")
        subj_path = os.path.join(DATA_ROOT, subj_id)
        mr_path = os.path.join(subj_path, "moved_mr.nii")
        ct_path = os.path.join(subj_path, "ct.nii")

        mr_nii = nib.load(mr_path)
        ct_nii = nib.load(ct_path)

        mr_data = mr_nii.get_fdata().transpose(2, 1, 0)
        ct_data = ct_nii.get_fdata().transpose(2, 1, 0)

        model = load_anatomix_model(ver, DEVICE)
        mr_feats = extract_features(model, mr_data, DEVICE)
        ct_feats = extract_features(model, ct_data, DEVICE)

        # Shared PCA
        C = mr_feats.shape[0]
        X_mr = mr_feats.reshape(C, -1).T
        X_ct = ct_feats.reshape(C, -1).T
        idx_mr = np.random.choice(X_mr.shape[0], 25000, replace=False)
        idx_ct = np.random.choice(X_ct.shape[0], 25000, replace=False)
        samples = np.concatenate([X_mr[idx_mr], X_ct[idx_ct]], axis=0)

        shared_pca_obj = PCA(n_components=3).fit(samples)
        mr_pca, _ = project_pca(mr_feats, shared_pca_obj)
        ct_pca, _ = project_pca(ct_feats, shared_pca_obj)

        global current_data
        current_data = {"mr_data": mr_data, "ct_data": ct_data, "mr_feats": mr_feats, "ct_feats": ct_feats, "mr_pca": mr_pca, "ct_pca": ct_pca, "z_idx": mr_data.shape[0] // 2}
        print("Done! Scroll down to visualize.")


load_btn.on_click(on_load_clicked)
display(widgets.HBox([ver_dropdown, region_dropdown, load_btn]), output)

Output()

In [4]:
def get_similarity_map(feats, seed_coord):
    C, D, H, W = feats.shape
    z, y, x = seed_coord
    z, y, x = min(int(z), D - 1), min(int(y), H - 1), min(int(x), W - 1)

    seed_vec = feats[:, z, y, x]
    X = feats.reshape(C, -1)

    seed_vec_norm = seed_vec / (np.linalg.norm(seed_vec) + 1e-8)
    X_norm = X / (np.linalg.norm(X, axis=0, keepdims=True) + 1e-8)

    sim = np.dot(seed_vec_norm, X_norm)
    return sim.reshape(D, H, W)


def visualize_interactive():
    if not current_data:
        print("Please load data first!")
        return

    mr_data = current_data["mr_data"]
    ct_data = current_data["ct_data"]
    mr_pca = current_data["mr_pca"]
    ct_pca = current_data["ct_pca"]
    mr_feats = current_data["mr_feats"]
    ct_feats = current_data["ct_feats"]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    z_slider = widgets.IntSlider(min=0, max=mr_data.shape[0] - 1, value=mr_data.shape[0] // 2, description="Z-slice:")

    titles = ["MRI", "CT", "Cosine Sim (MR Feats)", "MR PCA", "CT PCA", "Cosine Sim (CT Feats)"]
    ax_list = axes.flatten()
    ims = []

    def update_plots(z):
        # Clear existing content (better than creating new objects)
        for ax in ax_list:
            ax.clear()

        ims.clear()
        ims.append(ax_list[0].imshow(mr_data[z], cmap="gray"))
        ims.append(ax_list[1].imshow(ct_data[z], cmap="gray"))
        ims.append(ax_list[2].imshow(np.zeros_like(ct_data[z]), cmap="jet", vmin=0, vmax=1))
        ims.append(ax_list[3].imshow(mr_pca[z]))
        ims.append(ax_list[4].imshow(ct_pca[z]))
        ims.append(ax_list[5].imshow(np.zeros_like(ct_data[z]), cmap="jet", vmin=0, vmax=1))

        for i, title in enumerate(titles):
            ax_list[i].set_title(title)
            ax_list[i].axis("off")

        fig.canvas.draw_idle()

    def on_click(event):
        if event.inaxes != ax_list[1]:  # Only react to clicks on GT CT
            return

        y, x = int(event.ydata), int(event.xdata)
        z = z_slider.value

        # Clear previous markers
        for ax in ax_list:
            for artist in ax.collections:
                artist.remove()

        # Add markers
        ax_list[1].scatter(x, y, color="white", marker="x", s=100)
        ax_list[2].scatter(x, y, color="white", marker="x", s=100)
        ax_list[5].scatter(x, y, color="white", marker="x", s=100)

        # Compute sim
        sim_mr = get_similarity_map(mr_feats, (z, y, x))
        sim_ct = get_similarity_map(ct_feats, (z, y, x))

        ax_list[2].imshow(sim_mr[z], cmap="jet", vmin=0, vmax=1)
        ax_list[5].imshow(sim_ct[z], cmap="jet", vmin=0, vmax=1)

        fig.canvas.draw_idle()

    fig.canvas.mpl_connect("button_press_event", on_click)
    widgets.interactive(update_plots, z=z_slider)
    display(z_slider)
    update_plots(z_slider.value)


visualize_interactive()

Please load data first!
